# EX0005Cc · Exploratory validation A (post-pilot amendment)

**Explicit status: EXPLORATORY / NOT RUN.** The 32-item pilot scored 13/32 correct and 22/32 completed. The original v1 `NO_QUALIFIED_DISTRIBUTION` remains unchanged because its 75% completion rule was not met. The user then authorized disregarding that design rule for a separately documented exploratory continuation.

We select **original candidate A (4 main / 8 P1 / 8 P2 / 4 synthetic)**, not a newly invented candidate. Its pilot-weighted estimates are 9.5/24 correct and 16/24 complete. These are predictions, not validation results. No edits, Luna calls, optimizers, or historical EX0005C holdouts are allowed.

The validation pool was sealed before the 32-item pilot. Its identity/content hash is compared below against the uploaded receipt. **Run the one-shot validation only if you want this distinct exploratory experiment.** It does not modify any previous verdict.


In [ ]:
# 1. Use the same HF / PyTorch operating-room environment as the successful smoke test.
%pip -q install 'transformers==4.57.3' 'datasets==3.6.0' 'accelerate>=1.0,<2' 'safetensors>=0.5,<1'
import torch, transformers, datasets, time, platform, json
from pathlib import Path
assert torch.cuda.is_available(), 'STOP: CUDA unavailable'
assert 'L4' in torch.cuda.get_device_name(0), 'STOP: this independent AET specifies an L4'
assert torch.cuda.is_bf16_supported(), 'STOP: BF16 unavailable'
print('GPU:',torch.cuda.get_device_name(0))
print('versions:',torch.__version__,transformers.__version__,datasets.__version__)


### Verify the unaltered original source snapshots
The original dataset/scoring source is embedded and SHA-checked. No live evaluation occurs here.


In [ ]:
# 2. Materialize and verify source files locally.
from pathlib import Path
import hashlib,base64,sys,subprocess,os
source_dir=Path('/content/EX0005Cc_AET'); source_dir.mkdir(parents=True,exist_ok=True)
snapshots={'ex0005c_scorer.py': {'base64': 'IiIiRVgwMDA1QyB2MC4zIHByZWZsaWdodCBzY29yZXIuIE9mZmxpbmUgYW5kIHNpZGUtZWZmZWN0IGZyZWUuCgpEbyBub3QgY291bnQgdHJ1bmNhdGVkLCBtYWxmb3JtZWQsIG9yIHVucGFyc2VhYmxlIHJlc3BvbnNlcyBhcyBjb3JyZWN0LgpUaGUgZXZhbHVhdG9yIHJlY29yZHMgJ2ZpbmlzaGVkJyBhbmQgJ2NvbXBsZXRlZCcgc2VwYXJhdGVseS4KIiIiCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcywgYXNkaWN0CmZyb20gZGVjaW1hbCBpbXBvcnQgRGVjaW1hbCwgSW52YWxpZE9wZXJhdGlvbgppbXBvcnQgcmUKZnJvbSBjb2xsZWN0aW9ucyBpbXBvcnQgQ291bnRlcgppbXBvcnQgcmFuZG9tCgpGSU5BTCA9IHJlLmNvbXBpbGUociIoP20pXkZpbmFsIGFuc3dlcjpbIFx0XSooW15cclxuXSo/KVsgXHRdKiQiKQpOVU0gPSByZS5jb21waWxlKHIiWystXT8oPzpcZHsxLDN9KD86LFxkezN9KSt8XGQrKSg/OlwuXGQrKT8iKQoKCmRlZiBudW1lcmljKHZhbHVlKToKICAgICIiIkFjY2VwdCBvbmx5IHNpZ25lZCBudW1lcmljIHNjYWxhcnMgd2l0aCBvcHRpb25hbCBjb3JyZWN0bHkgcGxhY2VkIGNvbW1hcy4iIiIKICAgIHZhbHVlID0gdmFsdWUuc3RyaXAoKQogICAgaWYgbm90IE5VTS5mdWxsbWF0Y2godmFsdWUpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJpbnZhbGlkIG51bWVyaWMgYW5zd2VyOiB7dmFsdWUhcn0iKQogICAgdHJ5OgogICAgICAgIGQgPSBEZWNpbWFsKHZhbHVlLnJlcGxhY2UoJywnLCAnJykpCiAgICBleGNlcHQgSW52YWxpZE9wZXJhdGlvbiBhcyBleGM6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignaW52YWxpZCBkZWNpbWFsJykgZnJvbSBleGMKICAgIGlmIG5vdCBkLmlzX2Zpbml0ZSgpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ25vbmZpbml0ZSBkZWNpbWFsJykKICAgIHJldHVybiBkCgoKZGVmIHJlZmVyZW5jZShhbnN3ZXIpOgogICAgIiIiRXh0cmFjdCBHU004SydzIGF1dGhvcml0YXRpdmUgIyMjIyBmaW5hbCBudW1lcmljIGFuc3dlci4iIiIKICAgIGhpdHMgPSByZS5maW5kYWxsKHIiKD9tKV4jIyMjXHMqKFteXHJcbl0rKVxzKiQiLCBhbnN3ZXIpCiAgICBpZiBsZW4oaGl0cykgIT0gMToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdleHBlY3RlZCBleGFjdGx5IG9uZSAjIyMjIHJlZmVyZW5jZSBhbnN3ZXInKQogICAgcmV0dXJuIG51bWVyaWMoaGl0c1swXSkKCgpAZGF0YWNsYXNzCmNsYXNzIFNjb3JlOgogICAgZmluaXNoZWQ6IGJvb2wKICAgIGNvbXBsZXRlZDogYm9vbAogICAgY29ycmVjdDogYm9vbAogICAgcGFyc2VkOiBzdHIgfCBOb25lCiAgICBzdGF0dXM6IHN0cgoKICAgIGRlZiBqc29uKHNlbGYpOgogICAgICAgIHJldHVybiBhc2RpY3Qoc2VsZikKCgpkZWYgc2NvcmVfbWF0aChvdXRwdXQsIHRydXRoLCAqLCBmaW5pc2hlZCwgcmVxdWlyZV90aGluaz1UcnVlKToKICAgICIiIm91dHB1dCBtdXN0IGJlIHJhdyBnZW5lcmF0aW9uIHRleHQgKGV4Y2x1ZGluZyBwcm9tcHQpOyBmaW5pc2hlZCBpcyBFT1Mgc3RvcCwgbm90IGxlbmd0aCBjdXRvZmYuCgogICAgT25seSB0aGUgdGV4dCBhZnRlciB0aGUgY2xvc2VkIHRoaW5raW5nIHRhZyBjb3VudHMgYXMgYSBmaW5hbCBhbnN3ZXIuIE5vIGV4dHJhCiAgICBwcm9zZSBtYXkgZm9sbG93IEZpbmFsIGFuc3dlcjogLi4uIC4gVGhlIG1vZGVsIG5lZWQgbm90IHNob3cgY29ycmVjdCByZWFzb25pbmcuCiAgICAiIiIKICAgIGlmIG5vdCBmaW5pc2hlZDoKICAgICAgICByZXR1cm4gU2NvcmUoRmFsc2UsIEZhbHNlLCBGYWxzZSwgTm9uZSwgJ3RydW5jYXRlZF9vcl91bnRlcm1pbmF0ZWQnKQogICAgaWYgcmVxdWlyZV90aGluazoKICAgICAgICBpZiBvdXRwdXQuY291bnQoJzwvdGhpbms+JykgIT0gMSBvciBvdXRwdXQuY291bnQoJzx0aGluaz4nKSAhPSAxIG9yIG91dHB1dC5pbmRleCgnPHRoaW5rPicpID4gb3V0cHV0LmluZGV4KCc8L3RoaW5rPicpOgogICAgICAgICAgICByZXR1cm4gU2NvcmUoVHJ1ZSwgRmFsc2UsIEZhbHNlLCBOb25lLCAndW5jbG9zZWRfb3JfbXVsdGlwbGVfdGhpbmtpbmdfYmxvY2tzJykKICAgICAgICBhbnN3ZXJfcGFydCA9IG91dHB1dC5zcGxpdCgnPC90aGluaz4nLCAxKVsxXQogICAgZWxzZToKICAgICAgICBhbnN3ZXJfcGFydCA9IG91dHB1dAogICAgYW5zd2VyX3BhcnQgPSBhbnN3ZXJfcGFydC5yZXBsYWNlKCc8fGltX2VuZHw+JywgJycpLnN0cmlwKCkKICAgIG1hdGNoZXMgPSBsaXN0KEZJTkFMLmZpbmRpdGVyKGFuc3dlcl9wYXJ0KSkKICAgIGlmIGxlbihtYXRjaGVzKSAhPSAxOgogICAgICAgIHJldHVybiBTY29yZShUcnVlLCBGYWxzZSwgRmFsc2UsIE5vbmUsICdtaXNzaW5nX29yX211bHRpcGxlX2ZpbmFsX2Fuc3dlcnMnKQogICAgbWF0Y2ggPSBtYXRjaGVzWzBdCiAgICBpZiBhbnN3ZXJfcGFydFttYXRjaC5lbmQoKTpdLnN0cmlwKCk6CiAgICAgICAgcmV0dXJuIFNjb3JlKFRydWUsIEZhbHNlLCBGYWxzZSwgTm9uZSwgJ3RyYWlsaW5nX3RleHRfYWZ0ZXJfZmluYWwnKQogICAgdHJ5OgogICAgICAgIGdvdCA9IG51bWVyaWMobWF0Y2guZ3JvdXAoMSkpCiAgICAgICAgZ29sZCA9IG51bWVyaWMoc3RyKHRydXRoKSkKICAgIGV4Y2VwdCBWYWx1ZUVycm9yOgogICAgICAgIHJldHVybiBTY29yZShUcnVlLCBGYWxzZSwgRmFsc2UsIE5vbmUsICdpbnZhbGlkX251bWJlcicpCiAgICByZXR1cm4gU2NvcmUoVHJ1ZSwgVHJ1ZSwgZ290ID09IGdvbGQsIHN0cihnb3QpLCAndmFsaWRfY29tcGxldGUnKQoKCmRlZiBzY29yZV90ZXh0KG91dHB1dCwgdHJ1dGgsICosIGZpbmlzaGVkLCByZXF1aXJlX3RoaW5rPVRydWUpOgogICAgaWYgbm90IGZpbmlzaGVkOgogICAgICAgIHJldHVybiBTY29yZShGYWxzZSwgRmFsc2UsIEZhbHNlLCBOb25lLCAndHJ1bmNhdGVkX29yX3VudGVybWluYXRlZCcpCiAgICBpZiByZXF1aXJlX3RoaW5rOgogICAgICAgIGlmIG91dHB1dC5jb3VudCgnPC90aGluaz4nKSAhPSAxIG9yIG91dHB1dC5jb3VudCgnPHRoaW5rPicpICE9IDEgb3Igb3V0cHV0LmluZGV4KCc8dGhpbms+JykgPiBvdXRwdXQuaW5kZXgoJzwvdGhpbms+Jyk6CiAgICAgICAgICAgIHJldHVybiBTY29yZShUcnVlLCBGYWxzZSwgRmFsc2UsIE5vbmUsICd1bmNsb3NlZF9vcl9tdWx0aXBsZV90aGlua2luZ19ibG9ja3MnKQogICAgICAgIGFuc3dlcl9wYXJ0ID0gb3V0cHV0LnNwbGl0KCc8L3RoaW5rPicsIDEpWzFdCiAgICBlbHNlOgogICAgICAgIGFuc3dlcl9wYXJ0ID0gb3V0cHV0CiAgICBhbnN3ZXJfcGFydCA9IGFuc3dlcl9wYXJ0LnJlcGxhY2UoJzx8aW1fZW5kfD4nLCAnJykuc3RyaXAoKQogICAgbWF0Y2hlcyA9IGxpc3QoRklOQUwuZmluZGl0ZXIoYW5zd2VyX3BhcnQpKQogICAgaWYgbGVuKG1hdGNoZXMpICE9IDEgb3IgYW5zd2VyX3BhcnRbbWF0Y2hlc1swXS5lbmQoKTpdLnN0cmlwKCk6CiAgICAgICAgcmV0dXJuIFNjb3JlKFRydWUsIEZhbHNlLCBGYWxzZSwgTm9uZSwgJ21pc3Npbmdfb3JfZXh0cmFfZmluYWxfdGV4dCcpCiAgICB2YWx1ZSA9ICcgJy5qb2luKG1hdGNoZXNbMF0uZ3JvdXAoMSkubG93ZXIoKS5zcGxpdCgpKQogICAgaWYgbm90IHZhbHVlIG9yIGxlbih2YWx1ZSkgPiA4MDoKICAgICAgICByZXR1cm4gU2NvcmUoVHJ1ZSwgRmFsc2UsIEZhbHNlLCBOb25lLCAnaW52YWxpZF9maW5hbF90ZXh0JykKICAgIGV4cGVjdGVkID0gJyAnLmpvaW4oc3RyKHRydXRoKS5sb3dlcigpLnNwbGl0KCkpCiAgICByZXR1cm4gU2NvcmUoVHJ1ZSwgVHJ1ZSwgdmFsdWUgPT0gZXhwZWN0ZWQsIHZhbHVlLCAndmFsaWRfY29tcGxldGUnKQoKCmRlZiBkZWNvbXBvc2l0aW9uKGJhc2VsaW5lLCBlZGl0ZWQpOgogICAgIiIiUGFpcmVkIHVuaXRzOiBlYWNoIHJvdyBoYXMgeydjb21wbGV0ZWQnOmJvb2wsJ2NvcnJlY3QnOmJvb2x9OyBzYW1lIGl0ZW0gb3JkZXIuIiIiCiAgICBpZiBsZW4oYmFzZWxpbmUpICE9IGxlbihlZGl0ZWQpIG9yIG5vdCBiYXNlbGluZToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdub25lbXB0eSBlcXVhbC1sZW5ndGggcGFpcmVkIGRhdGEgcmVxdWlyZWQnKQogICAgYm90aCwgY2hhbmdlZCwgcjJ3LCB3MnIgPSAwLCAwLCAwLCAwCiAgICBmb3IgYiwgZSBpbiB6aXAoYmFzZWxpbmUsIGVkaXRlZCk6CiAgICAgICAgaWYgYlsnY29ycmVjdCddIGFuZCBub3QgYlsnY29tcGxldGVkJ10gb3IgZVsnY29ycmVjdCddIGFuZCBub3QgZVsnY29tcGxldGVkJ106CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ2NvcnJlY3QgYW5zd2VycyBjYW5ub3QgYmUgaW5jb21wbGV0ZScpCiAgICAgICAgZGlmZiA9IGludChlWydjb3JyZWN0J10pIC0gaW50KGJbJ2NvcnJlY3QnXSkKICAgICAgICBpZiBiWydjb21wbGV0ZWQnXSA9PSBlWydjb21wbGV0ZWQnXToKICAgICAgICAgICAgYm90aCArPSBkaWZmCiAgICAgICAgZWxzZToKICAgICAgICAgICAgY2hhbmdlZCArPSBkaWZmCiAgICAgICAgdzJyICs9IGRpZmYgPT0gMQogICAgICAgIHIydyArPSBkaWZmID09IC0xCiAgICByZXR1cm4gZGljdChuPWxlbihiYXNlbGluZSksIG5ldF90b3RhbD1ib3RoK2NoYW5nZWQsIGJvdGhfY29tcGxldGU9Ym90aCwKICAgICAgICAgICAgICAgIGNvbXBsZXRpb25fY2hhbmdlZD1jaGFuZ2VkLCB3cm9uZ190b19yaWdodD13MnIsIHJpZ2h0X3RvX3dyb25nPXIydywKICAgICAgICAgICAgICAgIGJhc2VsaW5lX2NvcnJlY3Q9c3VtKHhbJ2NvcnJlY3QnXSBmb3IgeCBpbiBiYXNlbGluZSksCiAgICAgICAgICAgICAgICBlZGl0ZWRfY29ycmVjdD1zdW0oeFsnY29ycmVjdCddIGZvciB4IGluIGVkaXRlZCkpCgoKZGVmIHBhaXJlZF9ib290c3RyYXBfY2koYmFzZWxpbmUsIGVkaXRlZCwgc2VlZD0yMDI2MTEyMiwgcmVwcz0xMDAwMCk6CiAgICAiIiJFeHBsb3JhdG9yeSBwZXJjZW50aWxlIENJIG9uIGZpeGVkIDQ4LWl0ZW0gcGFpcmVkIGFjY3VyYWN5IGRpZmZlcmVuY2UuIiIiCiAgICBpZiBsZW4oYmFzZWxpbmUpICE9IGxlbihlZGl0ZWQpIG9yIG5vdCBiYXNlbGluZToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdpbnZhbGlkIHBhaXJpbmcnKQogICAgZGlmZnMgPSBbaW50KGVbJ2NvcnJlY3QnXSkgLSBpbnQoYlsnY29ycmVjdCddKSBmb3IgYiwgZSBpbiB6aXAoYmFzZWxpbmUsIGVkaXRlZCldCiAgICBybmcgPSByYW5kb20uUmFuZG9tKHNlZWQpCiAgICBuID0gbGVuKGRpZmZzKQogICAgdmFsdWVzID0gc29ydGVkKHN1bShybmcuY2hvaWNlKGRpZmZzKSBmb3IgXyBpbiByYW5nZShuKSkvbiBmb3IgXyBpbiByYW5nZShyZXBzKSkKICAgIHJldHVybiAodmFsdWVzW2ludCgwLjAyNSpyZXBzKV0sIHZhbHVlc1ttaW4ocmVwcy0xLGludCgwLjk3NSpyZXBzKSldKQo=', 'sha256': 'ce0fe0b4633e9c3175889b763fe91fa25e231f5e0889d061e24ee28bd23649ed'}, 'ex0005cc_calibration.py': {'base64': 'IiIiRVgwMDA1Q2MgaW5kZXBlbmRlbnQgZGlmZmljdWx0eS1jYWxpYnJhdGlvbiBwaWxvdC4gTm8gbW9kZWwvaGVsZG91dCBhY2Nlc3MgaGVyZS4KCkFwcGxlIHJvd3MgYXJlIHJldHJpZXZlZCBvbmx5IGF0IHJ1bnRpbWUgKGxpY2Vuc2UgQ0MgQlktTkMtTkQgNC4wKTsgbm8gc291cmNlCnF1ZXN0aW9ucyBhcmUgZGlzdHJpYnV0ZWQgaW4gdGhpcyBtb2R1bGUuIFRoZSBoaXN0b3JpY2FsIEVYMDAwNUMgbWFuaWZlc3QgaXMKcmVhZC1vbmx5LCByZWZlcmVuY2VkIGJ5IFNIQSBhbmQgZXhjbHVkZWQgZnJvbSBzb3VyY2UtZmFtaWx5IHNlbGVjdGlvbi4KIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGpzb24KaW1wb3J0IG1hdGgKaW1wb3J0IHJhbmRvbQpmcm9tIGNvbGxlY3Rpb25zIGltcG9ydCBkZWZhdWx0ZGljdApmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKZnJvbSBleDAwMDVjX3Njb3JlciBpbXBvcnQgcmVmZXJlbmNlCgpSVU5fSUQgPSAnRVgwMDA1Q2MtQUVULTIwMjYtMTEtMjItdjEnCk1PREVMID0gJ1F3ZW4vUXdlbjMtMC42QicKTU9ERUxfUkVWID0gJ2MxODk5ZGUyODlhMDRkMTIxMDBkYjM3MGQ4MTQ4NWNkZjc1ZTQ3Y2EnCkFQUExFID0gJ2FwcGxlL0dTTS1TeW1ib2xpYycKQVBQTEVfUkVWID0gJzkzYjViMzc1OGQ5ZDk4NDFmZmU4MWQ2Y2QyYWUyYjAzMDY4NWIwNzgnCkhJU1RPUklDQUxfUkVDRUlQVF9TSEEyNTYgPSAnNmQwNzExMWJjYWY4MWFmMmExMjNmMTJlYTEyODg3YjNkMjIyZTE2MjM0N2IxMTFkN2VmMmQ5MmIwYTgwY2UwMScKIyBIaXN0b3JpY2FsIEVYMDAwNUMgaWRlbnRpZmllcnMgb25seS4gTmV2ZXIgbG9hZCBvbGQgaGVsZC1vdXQgcHJvbXB0L2Fuc3dlciB0ZXh0LgpFWENMVURFRF9URU1QTEFURSA9IGZyb3plbnNldChbOTYsMzQsNzIsNDcsNTQsMjQsMjcsMjYsOTMsNTYsNjcsMzMsODEsMywxMSwxOCw5MCw3Nyw0OCwxOSw3Myw2NCwxMCwzMF0pCkVYQ0xVREVEX0dTTV9PUklHID0gZnJvemVuc2V0KFsxMTgzLDExODQsNjA1LDEzMTYsOTE5LDg5Myw5ODAsOTEwLDkxMiw0NTMsMSwxMDIwLDExOTQsMTIwMCwyNDAsNDc4LDkwMiwxMTQyLDIyNCwxMyw3NiwxMjM4LDMzOCw2ODRdKQpTVFJBVEEgPSAoJ21haW4nLCdwMScsJ3AyJywnc3ludGhldGljJykKUElMT1RfTiA9IHsnbWFpbic6OCwncDEnOjgsJ3AyJzo4LCdzeW50aGV0aWMnOjh9CiMgT3JkZXJlZCBjYW5kaWRhdGUgbWl4dHVyZXMuIFRpZXMgZmF2b3IgZWFybGllciBtaXh0dXJlLiBObyBhZGRpdGlvbmFsIGNhbmRpZGF0ZXMgYWZ0ZXIgcGlsb3QuCk1JWFRVUkVTID0gKAogICAgKCdBJywgKDQsOCw4LDQpKSwKICAgICgnQicsICg0LDQsMTIsNCkpLAogICAgKCdDJywgKDAsOCw4LDgpKSwKICAgICgnRCcsICgwLDQsMTIsOCkpLAogICAgKCdFJywgKDAsNCw4LDEyKSksCiAgICAoJ0YnLCAoMCwwLDEyLDEyKSksCikKVEFSR0VUID0gMC41NQpBQ0NVUkFDWV9SQU5HRSA9ICgwLjMwLCAwLjc1KQpNSU5fQ09NUExFVEUgPSAwLjc1CgpkZWYgc2gob2JqKToKICAgIHJldHVybiBoYXNobGliLnNoYTI1Nihqc29uLmR1bXBzKG9iaixzb3J0X2tleXM9VHJ1ZSxlbnN1cmVfYXNjaWk9RmFsc2Usc2VwYXJhdG9ycz0oJywnLCc6JykpLmVuY29kZSgpKS5oZXhkaWdlc3QoKQoKZGVmIG9yZGVyZWQodmFsdWVzLCBuYW1lc3BhY2UpOgogICAgcmV0dXJuIHNvcnRlZCh2YWx1ZXMsa2V5PWxhbWJkYSB2OiBoYXNobGliLnNoYTI1NihmJ3tSVU5fSUR9fHtuYW1lc3BhY2V9fHt2fScuZW5jb2RlKCkpLmhleGRpZ2VzdCgpKQoKZGVmIHByb21wdF9pZChyb3cpOgogICAgcmV0dXJuIChpbnQocm93WydpZCddKSxpbnQocm93WydvcmlnaW5hbF9pZCddKSxpbnQocm93WydpbnN0YW5jZSddKSkKCmRlZiBzeW50aGV0aWNfb25lKGZhbWlseTogc3RyLCBpbmRleDogaW50LCBwaGFzZTogc3RyKToKICAgIHJuZyA9IHJhbmRvbS5SYW5kb20oaW50KHNoKFtSVU5fSUQsJ3N5bnRoZXRpYycscGhhc2UsZmFtaWx5LGluZGV4XSlbOjE2XSwxNikpCiAgICBpZiBmYW1pbHk9PSdpbnZlbnRvcnknOgogICAgICAgICMgRW5zdXJlIGV4YWN0IGludGVncmFsIHNoaXBtZW50IGFuZCBhIHJlYWxpc3RpYyByZW1haW5kZXIuCiAgICAgICAgY2FydG9ucyA9IHJuZy5yYW5kcmFuZ2UoOCwxOSkKICAgICAgICB1bml0cyA9IHJuZy5yYW5kcmFuZ2UoOSwyMCkKICAgICAgICBleHRyYXMgPSBybmcucmFuZHJhbmdlKDMsMTApCiAgICAgICAgZXh0cmFfdW5pdHMgPSBybmcucmFuZHJhbmdlKDYsMTYpCiAgICAgICAgZiA9IHJuZy5jaG9pY2UoWzIsMyw0XSkKICAgICAgICB3aG9sZSA9IGNhcnRvbnMqdW5pdHMgKyBleHRyYXMqZXh0cmFfdW5pdHMKICAgICAgICAjIEFkZCBhIHNtYWxsIGludGVnZXIgcmVtYWluZGVyIHRvIG1ha2UgdGhlIGludmVudG9yeSBleGFjdGx5IGRpdmlzaWJsZSBieSBmLgogICAgICAgIGFkanVzdG1lbnRzID0gKC13aG9sZSklZgogICAgICAgIHJlbW92ZWQgPSBybmcucmFuZHJhbmdlKDIsNikKICAgICAgICByZW1vdmVfdW5pdHMgPSBybmcucmFuZHJhbmdlKDIsOCkKICAgICAgICByZXN0b2NrID0gcm5nLnJhbmRyYW5nZSg3LDIyKQogICAgICAgIGFuc3dlciA9ICh3aG9sZSthZGp1c3RtZW50cykqKGYtMSkvL2YgLSByZW1vdmVkKnJlbW92ZV91bml0cyArIHJlc3RvY2sKICAgICAgICBxPShmJ0Egd29ya3Nob3Agc3RvY2tlZCB7Y2FydG9uc30gY2FydG9ucyB3aXRoIHt1bml0c30gZ2VhcnMgZWFjaCwgdGhlbiB7ZXh0cmFzfSBjYXJ0b25zIHdpdGggJwogICAgICAgICAgIGYne2V4dHJhX3VuaXRzfSBnZWFycyBlYWNoLCBhbmQgZm91bmQge2FkanVzdG1lbnRzfSBsb29zZSBnZWFycy4gSXQgc2hpcHBlZCBleGFjdGx5IDEve2Z9ICcKICAgICAgICAgICBmJ29mIGl0cyBlbnRpcmUgc3RvY2suIEFmdGVyIHNoaXBwaW5nLCB7cmVtb3ZlZH0gcmVwYWlyIGtpdHMgdXNlZCB7cmVtb3ZlX3VuaXRzfSBnZWFycyBlYWNoLCAnCiAgICAgICAgICAgZidhbmQgYSBsYXRlIGRlbGl2ZXJ5IGJyb3VnaHQge3Jlc3RvY2t9IGdlYXJzLiBIb3cgbWFueSBnZWFycyByZW1haW4/JykKICAgICAgICBjaGVjayA9IHdob2xlK2FkanVzdG1lbnRzLSh3aG9sZSthZGp1c3RtZW50cykvL2YtcmVtb3ZlZCpyZW1vdmVfdW5pdHMrcmVzdG9jawogICAgZWxpZiBmYW1pbHk9PSd0aWNrZXRzJzoKICAgICAgICBhZHVsdHM9cm5nLnJhbmRyYW5nZSgyNCw0Nyk7IGNoaWxkcmVuPXJuZy5yYW5kcmFuZ2UoMTMsMzApCiAgICAgICAgYWR1bHRfcHJpY2U9cm5nLnJhbmRyYW5nZSg5LDIyKTsgY2hpbGRfcHJpY2U9cm5nLnJhbmRyYW5nZSg1LDEwKQogICAgICAgIHJlZnVuZF9hPXJuZy5yYW5kcmFuZ2UoMiw3KTsgcmVmdW5kX2M9cm5nLnJhbmRyYW5nZSgyLDYpCiAgICAgICAgY29zdHM9cm5nLnJhbmRyYW5nZSgyMiw0OSk7IGJvbnVzPXJuZy5yYW5kcmFuZ2UoOSwyMykKICAgICAgICBhbnN3ZXI9KGFkdWx0cy1yZWZ1bmRfYSkqYWR1bHRfcHJpY2UrKGNoaWxkcmVuLXJlZnVuZF9jKSpjaGlsZF9wcmljZS1jb3N0cytib251cwogICAgICAgIHE9KGYnQSBjbHViIHNvbGQge2FkdWx0c30gYWR1bHQgdGlja2V0cyBmb3IgJHthZHVsdF9wcmljZX0gZWFjaCBhbmQge2NoaWxkcmVufSBjaGlsZCB0aWNrZXRzICcKICAgICAgICAgICBmJ2ZvciAke2NoaWxkX3ByaWNlfSBlYWNoLiBCZWZvcmUgdGhlIGV2ZW50IGl0IGZ1bGx5IHJlZnVuZGVkIHtyZWZ1bmRfYX0gYWR1bHQgdGlja2V0cyAnCiAgICAgICAgICAgZidhbmQge3JlZnVuZF9jfSBjaGlsZCB0aWNrZXRzLiBJdCB0aGVuIHBhaWQgJHtjb3N0c30gdG8gcmVudCBlcXVpcG1lbnQgYW5kIHJlY2VpdmVkICcKICAgICAgICAgICBmJ2EgJHtib251c30gY29udHJpYnV0aW9uLiBXaGF0IGlzIGl0cyBuZXQgYW1vdW50IG9mIG1vbmV5IGluIGRvbGxhcnM/JykKICAgICAgICBjaGVjaz1hZHVsdHMqYWR1bHRfcHJpY2UrY2hpbGRyZW4qY2hpbGRfcHJpY2UtcmVmdW5kX2EqYWR1bHRfcHJpY2UtcmVmdW5kX2MqY2hpbGRfcHJpY2UtY29zdHMrYm9udXMKICAgIGVsaWYgZmFtaWx5PT0ncmF0ZXMnOgogICAgICAgIHRlYW1zPXJuZy5yYW5kcmFuZ2UoMyw3KTsgZHJpdmVycz1ybmcucmFuZHJhbmdlKDMsNykKICAgICAgICBwZXI9cm5nLnJhbmRyYW5nZSg4LDE5KTsgZGF5cz1ybmcucmFuZHJhbmdlKDMsOCkKICAgICAgICB1c2VkPXJuZy5yYW5kcmFuZ2UoMjAsNDgpOyBsb3NzPXJuZy5yYW5kcmFuZ2UoNiwxNyk7IHJlc2VydmU9cm5nLnJhbmRyYW5nZSgxMywzMSkKICAgICAgICBhbnN3ZXI9dGVhbXMqZHJpdmVycypwZXIqZGF5cyAtIHVzZWQqZGF5cyAtIGxvc3MgKyByZXNlcnZlCiAgICAgICAgcT0oZidBIGRlcG90IGhhcyB7dGVhbXN9IHRlYW1zIHdpdGgge2RyaXZlcnN9IGRyaXZlcnMgb24gZWFjaCB0ZWFtLiBFdmVyeSBkcml2ZXIgZGVsaXZlcnMgJwogICAgICAgICAgIGYne3Blcn0gcGFja2FnZXMgcGVyIGRheSBmb3Ige2RheXN9IGRheXMuIEVhY2ggZGF5IHRoZSBkZXBvdCB1c2VzIHt1c2VkfSBkZWxpdmVyZWQgJwogICAgICAgICAgIGYncGFja2FnZXMgZm9yIHRlc3RpbmcuIEFub3RoZXIge2xvc3N9IHBhY2thZ2VzIHdlcmUgZGFtYWdlZCBhdCB0aGUgZW5kLCBidXQgdGhlIGRlcG90ICcKICAgICAgICAgICBmJ2Fsc28gcmVjZWl2ZXMge3Jlc2VydmV9IHJlc2VydmUgcGFja2FnZXMuIEhvdyBtYW55IHVzYWJsZSBwYWNrYWdlcyBkb2VzIHRoZSBkZXBvdCBoYXZlPycpCiAgICAgICAgY2hlY2s9KHRlYW1zKmRyaXZlcnMqcGVyLXVzZWQpKmRheXMtbG9zcytyZXNlcnZlCiAgICBlbGlmIGZhbWlseT09J2VxdWF0aW9ucyc6CiAgICAgICAgYj1ybmcucmFuZHJhbmdlKDgsMTkpOyBtdWx0PXJuZy5jaG9pY2UoWzIsMyw0XSk7IG1vcmU9cm5nLnJhbmRyYW5nZSg0LDE1KQogICAgICAgIG1vdmVkPXJuZy5yYW5kcmFuZ2UoMiw4KTsgam9pbj1ybmcucmFuZHJhbmdlKDEsOCkKICAgICAgICB0b3RhbD1iICsgbXVsdCpiICsgKGIrbW9yZSkKICAgICAgICBhbnN3ZXI9KGIrbW9yZSktbW92ZWQram9pbgogICAgICAgIHE9KGYnQmVmb3JlIGEgd29ya3Nob3AsIHRocmVlIGNsYXNzcm9vbXMgQSwgQiBhbmQgQyBjb250YWluIHt0b3RhbH0gc3R1ZGVudHMgYWx0b2dldGhlci4gJwogICAgICAgICAgIGYnUm9vbSBBIGhhcyBleGFjdGx5IHttdWx0fSB0aW1lcyBhcyBtYW55IHN0dWRlbnRzIGFzIHJvb20gQjsgcm9vbSBDIGhhcyBleGFjdGx5ICcKICAgICAgICAgICBmJ3ttb3JlfSBtb3JlIHN0dWRlbnRzIHRoYW4gcm9vbSBCLiBMYXRlciB7bW92ZWR9IHN0dWRlbnRzIGxlYXZlIHJvb20gQyBhbmQgJwogICAgICAgICAgIGYne2pvaW59IHN0dWRlbnRzIGpvaW4gcm9vbSBDLiBIb3cgbWFueSBzdHVkZW50cyBhcmUgaW4gQyBhZnRlcndhcmQ/JykKICAgICAgICByZWNvdmVyZWQ9KHRvdGFsLW1vcmUpLy8obXVsdCsyKQogICAgICAgIGNoZWNrPXJlY292ZXJlZCttb3JlLW1vdmVkK2pvaW4KICAgIGVsc2U6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigndW5rbm93biBzeW50aGV0aWMgZmFtaWx5JykKICAgIGFzc2VydCBpc2luc3RhbmNlKGFuc3dlcixpbnQpIGFuZCBhbnN3ZXI9PWNoZWNrIGFuZCBhbnN3ZXI+PTAKICAgIHJldHVybiB7J2lkJzpmJ3N5bnRoZXRpYzp7cGhhc2V9OntmYW1pbHl9OntpbmRleH0nLAogICAgICAgICAgICAnc3RyYXR1bSc6J3N5bnRoZXRpYycsJ2tpbmQnOidtYXRoJywnZmFtaWx5JzpmJ3N5bnRoZXRpYzp7ZmFtaWx5fScsCiAgICAgICAgICAgICdxdWVzdGlvbic6cSwndHJ1dGgnOnN0cihhbnN3ZXIpLCdvcmFjbGUnOid0d29fZm9ybXVsYV9weXRob24nLAogICAgICAgICAgICAncGFyYW1ldGVyc19oYXNoJzpzaChbZmFtaWx5LHBoYXNlLGluZGV4XSl9CgpkZWYgYnVpbGRfc3ludGhldGljKHBoYXNlOiBzdHIsIGNvdW50OiBpbnQpOgogICAgZmFtaWxpZXM9WydpbnZlbnRvcnknLCd0aWNrZXRzJywncmF0ZXMnLCdlcXVhdGlvbnMnXQogICAgcmV0dXJuIFtzeW50aGV0aWNfb25lKGZhbWlsaWVzW2klbGVuKGZhbWlsaWVzKV0saS8vbGVuKGZhbWlsaWVzKSxwaGFzZSkKICAgICAgICAgICAgZm9yIGkgaW4gcmFuZ2UoY291bnQpXQoKZGVmIGxvYWRfYXBwbGUoKToKICAgIGZyb20gZGF0YXNldHMgaW1wb3J0IGxvYWRfZGF0YXNldAogICAgcmV0dXJuIHtuYW1lOmxpc3QobG9hZF9kYXRhc2V0KEFQUExFLG5hbWU9bmFtZSxzcGxpdD0ndGVzdCcscmV2aXNpb249QVBQTEVfUkVWKSkKICAgICAgICAgICAgZm9yIG5hbWUgaW4gKCdtYWluJywncDEnLCdwMicpfQoKZGVmIGNvbnN0cnVjdF9hcHBsZShjYWxpYnJhdGlvbnM6IGRpY3QpOgogICAgIiIiRGlzam9pbnQgR1NNIHNvdXJjZSBmYW1pbGllcyBhY3Jvc3MgcGlsb3QvdmFsaWRhdGlvbiBhbmQgYWxsIHRocmVlIEFwcGxlIHZhcmlhbnRzLiIiIgogICAgcG9vbD17fQogICAgZm9yIHN0cmF0dW0scm93cyBpbiBjYWxpYnJhdGlvbnMuaXRlbXMoKToKICAgICAgICBhc3NlcnQgc3RyYXR1bSBpbiAoJ21haW4nLCdwMScsJ3AyJykKICAgICAgICBmYW1pbGllcz1kZWZhdWx0ZGljdChsaXN0KQogICAgICAgIGZvciByb3cgaW4gcm93czoKICAgICAgICAgICAga2V5cz0oJ2lkJywnaW5zdGFuY2UnLCdvcmlnaW5hbF9pZCcsJ3F1ZXN0aW9uJywnYW5zd2VyJywnb3JpZ2luYWxfcXVlc3Rpb24nKQogICAgICAgICAgICBpZiBhbnkoayBub3QgaW4gcm93IGZvciBrIGluIGtleXMpOiByYWlzZSBWYWx1ZUVycm9yKGYnQXBwbGUgc2NoZW1hIG1pc21hdGNoIHtzdHJhdHVtfScpCiAgICAgICAgICAgIHRlbXBsYXRlLG9yaWcsaW5zdD1wcm9tcHRfaWQocm93KQogICAgICAgICAgICBpZiB0ZW1wbGF0ZSBpbiBFWENMVURFRF9URU1QTEFURSBvciBvcmlnIGluIEVYQ0xVREVEX0dTTV9PUklHOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgZmFtaWxpZXNbKHRlbXBsYXRlLG9yaWcpXS5hcHBlbmQocm93KQogICAgICAgIHBvb2xbc3RyYXR1bV09ZmFtaWxpZXMKICAgICAgICBpZiBsZW4oZmFtaWxpZXMpPDMyOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYnaW5zdWZmaWNpZW50IGVsaWdpYmxlIEFwcGxlIHNvdXJjZSBmYW1pbGllcyBpbiB7c3RyYXR1bX06IHtsZW4oZmFtaWxpZXMpfScpCiAgICB1c2VkPXNldCgpCiAgICBwaWxvdD17bmFtZTpbXSBmb3IgbmFtZSBpbiBTVFJBVEF9CiAgICB2YWxpZGF0ZT17bmFtZTpbXSBmb3IgbmFtZSBpbiBTVFJBVEF9CiAgICBmb3Igc3RhZ2UscmVxdWlyZWQgaW4gWygncGlsb3QnLFBJTE9UX04pLCgndmFsaWRhdGlvbicseydtYWluJzo0LCdwMSc6OCwncDInOjEyfSldOgogICAgICAgIGZvciBzdHJhdHVtIGluICgnbWFpbicsJ3AxJywncDInKToKICAgICAgICAgICAgZWxpZ2libGU9b3JkZXJlZChwb29sW3N0cmF0dW1dLGYne3N0YWdlfXx7c3RyYXR1bX18ZmFtaWx5JykKICAgICAgICAgICAgY291bnQ9MAogICAgICAgICAgICBmb3IgZmFtaWx5IGluIGVsaWdpYmxlOgogICAgICAgICAgICAgICAgXyxvcmlnPWZhbWlseQogICAgICAgICAgICAgICAgaWYgb3JpZyBpbiB1c2VkOiBjb250aW51ZQogICAgICAgICAgICAgICAgaW5zdGFuY2VzPXNvcnRlZChwb29sW3N0cmF0dW1dW2ZhbWlseV0sa2V5PWxhbWJkYSByOmhhc2hsaWIuc2hhMjU2KAogICAgICAgICAgICAgICAgICAgIGYne1JVTl9JRH18e3N0YWdlfXx7c3RyYXR1bX18aW5zdGFuY2V8e3JbImluc3RhbmNlIl19Jy5lbmNvZGUoKSkuaGV4ZGlnZXN0KCkpCiAgICAgICAgICAgICAgICByb3c9aW5zdGFuY2VzWzBdCiAgICAgICAgICAgICAgICByZWNvcmQ9eydpZCc6ZidhcHBsZTp7c3RyYXR1bX06e3Jvd1siaWQiXX06e3Jvd1siaW5zdGFuY2UiXX0nLAogICAgICAgICAgICAgICAgICAgICAgICAnc3RyYXR1bSc6c3RyYXR1bSwnZmFtaWx5JzpmJ2dzbThrX3Rlc3Q6e29yaWd9JywKICAgICAgICAgICAgICAgICAgICAgICAgJ2tpbmQnOidtYXRoJywncXVlc3Rpb24nOnJvd1sncXVlc3Rpb24nXSwKICAgICAgICAgICAgICAgICAgICAgICAgJ3RydXRoJzpzdHIocmVmZXJlbmNlKHJvd1snYW5zd2VyJ10pKSwKICAgICAgICAgICAgICAgICAgICAgICAgJ29yYWNsZSc6J2FwcGxlX2JlbmNobWFya19rZXknLCd0ZW1wbGF0ZV9pZCc6aW50KHJvd1snaWQnXSksCiAgICAgICAgICAgICAgICAgICAgICAgICdvcmlnaW5hbF9pZCc6b3JpZywnaW5zdGFuY2UnOmludChyb3dbJ2luc3RhbmNlJ10pLAogICAgICAgICAgICAgICAgICAgICAgICAnc291cmNlJzpmJ3tBUFBMRX06e3N0cmF0dW19QHtBUFBMRV9SRVZ9J30KICAgICAgICAgICAgICAgIChwaWxvdCBpZiBzdGFnZT09J3BpbG90JyBlbHNlIHZhbGlkYXRlKVtzdHJhdHVtXS5hcHBlbmQocmVjb3JkKQogICAgICAgICAgICAgICAgdXNlZC5hZGQob3JpZyk7IGNvdW50Kz0xCiAgICAgICAgICAgICAgICBpZiBjb3VudD09cmVxdWlyZWRbc3RyYXR1bV06IGJyZWFrCiAgICAgICAgICAgIGlmIGNvdW50ICE9IHJlcXVpcmVkW3N0cmF0dW1dOgogICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmJ2NvdWxkIG5vdCByZXNlcnZlIHtyZXF1aXJlZFtzdHJhdHVtXX0gdW5pcXVlIGZhbWlsaWVzIGZvciB7c3RhZ2V9L3tzdHJhdHVtfTsgZ290IHtjb3VudH0nKQogICAgcGlsb3RbJ3N5bnRoZXRpYyddPWJ1aWxkX3N5bnRoZXRpYygncGlsb3QnLDgpCiAgICB2YWxpZGF0ZVsnc3ludGhldGljJ109YnVpbGRfc3ludGhldGljKCd2YWxpZGF0aW9uJywxMikKICAgIGZsYXRfcGlsb3Q9W3IgZm9yIHN0cmF0dW0gaW4gU1RSQVRBIGZvciByIGluIHBpbG90W3N0cmF0dW1dXQogICAgYXNzZXJ0IGxlbihmbGF0X3BpbG90KT09MzIKICAgIGFzc2VydCBsZW4oc2V0KHJbJ2ZhbWlseSddIGZvciByIGluIGZsYXRfcGlsb3QgaWYgclsnc3RyYXR1bSddIT0nc3ludGhldGljJykpPT0yNAogICAgYXNzZXJ0IG5vdCAoc2V0KHJbJ2ZhbWlseSddIGZvciByIGluIGZsYXRfcGlsb3QgaWYgclsnc3RyYXR1bSddIT0nc3ludGhldGljJykgJgogICAgICAgICAgICAgICAgc2V0KHJbJ2ZhbWlseSddIGZvciBuYW1lIGluICgnbWFpbicsJ3AxJywncDInKSBmb3IgciBpbiB2YWxpZGF0ZVtuYW1lXSkpCiAgICByZXR1cm4gcGlsb3QsIHZhbGlkYXRlCgpkZWYgc2VsZWN0X2Rpc3RyaWJ1dGlvbihyZXN1bHRzKToKICAgICIiIlBpbG90IHJlc3VsdHMgY29udGFpbiB7J3N0cmF0dW0nLCdzY29yZSc6eydjb21wbGV0ZWQnLCdjb3JyZWN0J30sJ2lkJ30uCgogICAgU2VsZWN0IG9uICpzdHJhdHVtIGF2ZXJhZ2VzKiwgbmV2ZXIgb24gd2hpY2ggaW5kaXZpZHVhbCBjYWxpYnJhdGlvbiBpdGVtcyB3ZXJlIHdyb25nLgogICAgTm8gaHlwZXJwYXJhbWV0ZXIgcmVwYWlyIG9yIHBvc3Qtc2NvcmUgZXh0cmEgbWl4dHVyZSBwcm9wb3NhbHMuCiAgICAiIiIKICAgIGlmIGxlbihyZXN1bHRzKSE9MzIgb3IgbGVuKHtyWydpZCddIGZvciByIGluIHJlc3VsdHN9KSE9MzI6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignbXVzdCBzdXBwbHkgZXhhY3RseSB0aGUgcHJlY29tbWl0dGVkIDMyIHVuaXF1ZSBwaWxvdCBvdXRwdXRzJykKICAgIHN0cmF0YT17fQogICAgZm9yIG5hbWUgaW4gU1RSQVRBOgogICAgICAgIHJvd3M9W3IgZm9yIHIgaW4gcmVzdWx0cyBpZiByWydzdHJhdHVtJ109PW5hbWVdCiAgICAgICAgaWYgbGVuKHJvd3MpIT04OiByYWlzZSBWYWx1ZUVycm9yKGYnbWlzc2luZyBvciBkdXBsaWNhdGUge25hbWV9IHN0cmF0dW0nKQogICAgICAgIGlmIG5vdCBhbGwoaXNpbnN0YW5jZShyWydzY29yZSddWydjb3JyZWN0J10sYm9vbCkgYW5kCiAgICAgICAgICAgICAgICAgICBpc2luc3RhbmNlKHJbJ3Njb3JlJ11bJ2NvbXBsZXRlZCddLGJvb2wpIGZvciByIGluIHJvd3MpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdpbnZhbGlkIHNjb3JlZCB2YWx1ZXMnKQogICAgICAgIGlmIGFueShyWydzY29yZSddWydjb3JyZWN0J10gYW5kIG5vdCByWydzY29yZSddWydjb21wbGV0ZWQnXSBmb3IgciBpbiByb3dzKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignaW5jb25zaXN0ZW50IGNvcnJlY3RuZXNzL2NvbXBsZXRpb24nKQogICAgICAgIHN0cmF0YVtuYW1lXT17J24nOjgsJ2NvcnJlY3QnOnN1bShyWydzY29yZSddWydjb3JyZWN0J10gZm9yIHIgaW4gcm93cyksCiAgICAgICAgICAgICAgICAgICAgICAnY29tcGxldGUnOnN1bShyWydzY29yZSddWydjb21wbGV0ZWQnXSBmb3IgciBpbiByb3dzKX0KICAgIHN1bW1hcnk9W10KICAgIGZvciBuYW1lLHdlaWdodHMgaW4gTUlYVFVSRVM6CiAgICAgICAgcHJlZGljdGVkX2NvcnJlY3Q9c3VtKHdlaWdodHNbal0qc3RyYXRhW3NdWydjb3JyZWN0J10vOCBmb3IgaixzIGluIGVudW1lcmF0ZShTVFJBVEEpKQogICAgICAgIHByZWRpY3RlZF9jb21wbGV0ZT1zdW0od2VpZ2h0c1tqXSpzdHJhdGFbc11bJ2NvbXBsZXRlJ10vOCBmb3IgaixzIGluIGVudW1lcmF0ZShTVFJBVEEpKQogICAgICAgIHA9cHJlZGljdGVkX2NvcnJlY3QvMjQ7IGM9cHJlZGljdGVkX2NvbXBsZXRlLzI0CiAgICAgICAgZWxpZ2libGUgPSBBQ0NVUkFDWV9SQU5HRVswXTw9cDw9QUNDVVJBQ1lfUkFOR0VbMV0gYW5kIGM+PU1JTl9DT01QTEVURQogICAgICAgIHN1bW1hcnkuYXBwZW5kKHsnbWl4dHVyZSc6bmFtZSwnd2VpZ2h0cyc6ZGljdCh6aXAoU1RSQVRBLHdlaWdodHMpKSwKICAgICAgICAgICAgICAgICAgICAgICAgJ3ByZWRpY3RlZF9jb3JyZWN0X29mXzI0JzpwcmVkaWN0ZWRfY29ycmVjdCwKICAgICAgICAgICAgICAgICAgICAgICAgJ3ByZWRpY3RlZF9hY2N1cmFjeSc6cCwncHJlZGljdGVkX2NvbXBsZXRpb24nOmMsCiAgICAgICAgICAgICAgICAgICAgICAgICdlbGlnaWJsZSc6ZWxpZ2libGV9KQogICAgY2FuZGlkYXRlcz1beCBmb3IgeCBpbiBzdW1tYXJ5IGlmIHhbJ2VsaWdpYmxlJ11dCiAgICBpZiBjYW5kaWRhdGVzOgogICAgICAgIGNob2ljZT1taW4oY2FuZGlkYXRlcyxrZXk9bGFtYmRhIHg6KGFicyh4WydwcmVkaWN0ZWRfYWNjdXJhY3knXS1UQVJHRVQpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIG5leHQoaiBmb3IgaiwobixfKSBpbiBlbnVtZXJhdGUoTUlYVFVSRVMpIGlmIHhbJ21peHR1cmUnXT09bikpKQogICAgICAgIGRpc3Bvc2l0aW9uPSdQUk9WSVNJT05BTF9ESVNUUklCVVRJT05fU0VMRUNURUQnCiAgICBlbHNlOgogICAgICAgIGNob2ljZT1Ob25lCiAgICAgICAgZGlzcG9zaXRpb249J05PX1FVQUxJRklFRF9ESVNUUklCVVRJT04nCiAgICByZXR1cm4geydwcm90b2NvbCc6UlVOX0lELCdwaWxvdCc6c3RyYXRhLCdjYW5kaWRhdGVzJzpzdW1tYXJ5LAogICAgICAgICAgICAnY2hvaWNlJzpjaG9pY2UsJ2Rpc3Bvc2l0aW9uJzpkaXNwb3NpdGlvbiwKICAgICAgICAgICAgJ2NhdmVhdCc6J09ubHkgOCBwaWxvdCBvdXRwdXRzIHBlciBzdHJhdHVtOyB0aGlzIGlzIGEgbW9kZWwtYmFzZWQgcHJvdmlzaW9uYWwgbWl4dHVyZSwgbm90IHZhbGlkYXRlZCBnZW5lcmFsaXphdGlvbi4nfQoKZGVmIHdyaXRlX3F1YWxpZmljYXRpb25fcGFjayhyb290LCBwaWxvdCwgdmFsaWRhdGlvbik6CiAgICByb290PVBhdGgocm9vdCk7cm9vdC5ta2RpcihwYXJlbnRzPVRydWUsZXhpc3Rfb2s9VHJ1ZSkKICAgIHBpbG90X3Jvd3M9W3IgZm9yIHMgaW4gU1RSQVRBIGZvciByIGluIHBpbG90W3NdXQogICAgIyBWYWxpZGF0aW9uIHNvdXJjZSBwb29sIG1heSBiZSBnZW5lcmF0ZWQsIGFuZCBJRHMgc2VhbGVkLCBidXQgbmV2ZXIgc2NvcmVkIGhlcmUuCiAgICBtZXRhPXsncHJvdG9jb2wnOlJVTl9JRCwnc3RhdHVzJzonQ0FMSUJSQVRJT05fUFJFUEFSRUQnLAogICAgICAgICAgJ2hpc3RvcmljYWxfcmVjZWlwdF9zaGEyNTYnOkhJU1RPUklDQUxfUkVDRUlQVF9TSEEyNTYsCiAgICAgICAgICAnbW9kZWwnOk1PREVMLCdtb2RlbF9yZXZpc2lvbic6TU9ERUxfUkVWLAogICAgICAgICAgJ2FwcGxlX3JldmlzaW9uJzpBUFBMRV9SRVYsCiAgICAgICAgICAncGlsb3RfaWRzJzpbclsnaWQnXSBmb3IgciBpbiBwaWxvdF9yb3dzXSwKICAgICAgICAgICd2YWxpZGF0aW9uX3Bvb2xfaWRzJzpbclsnaWQnXSBmb3IgcyBpbiBTVFJBVEEgZm9yIHIgaW4gdmFsaWRhdGlvbltzXV0sCiAgICAgICAgICAncGlsb3RfY29udGVudF9oYXNoJzpzaChwaWxvdF9yb3dzKSwKICAgICAgICAgICd2YWxpZGF0aW9uX3Bvb2xfaGFzaCc6c2godmFsaWRhdGlvbil9CiAgICAocm9vdC8ncGlsb3QuanNvbicpLndyaXRlX3RleHQoanNvbi5kdW1wcyhwaWxvdF9yb3dzLGluZGVudD0yLGVuc3VyZV9hc2NpaT1GYWxzZSkpCiAgICAocm9vdC8ndmFsaWRhdGlvbl9wb29sX0RPX05PVF9TQ09SRS5qc29uJykud3JpdGVfdGV4dChqc29uLmR1bXBzKHZhbGlkYXRpb24saW5kZW50PTIsZW5zdXJlX2FzY2lpPUZhbHNlKSkKICAgIChyb290LydtYW5pZmVzdC5qc29uJykud3JpdGVfdGV4dChqc29uLmR1bXBzKG1ldGEsaW5kZW50PTIpKQogICAgcmV0dXJuIG1ldGEKCmRlZiB2YWxpZGF0ZV9zZWxlY3Rpb24oY2hvaWNlLCB2YWxpZGF0aW9uKToKICAgICIiIlNlYWwgMjQgZGlzam9pbnQgcXVhbGlmaWNhdGlvbiBJRHMgYmFzZWQgb24gc3RyYXR1bSB3ZWlnaHRzLCBuZXZlciBwaWxvdCBvdXRjb21lcy4iIiIKICAgIGlmIG5vdCBjaG9pY2U6IHJldHVybiBOb25lCiAgICBwaWNrZWQ9W10KICAgIGZvciBzIGluIFNUUkFUQToKICAgICAgICBuPWNob2ljZVsnd2VpZ2h0cyddW3NdCiAgICAgICAgc2VsZWN0ZWQ9dmFsaWRhdGlvbltzXVs6bl0KICAgICAgICBpZiBsZW4oc2VsZWN0ZWQpIT1uOnJhaXNlIFZhbHVlRXJyb3IoZidpbnN1ZmZpY2llbnQgZnJlc2gge3N9IHZhbGlkYXRpb24gcm93cycpCiAgICAgICAgcGlja2VkLmV4dGVuZChzZWxlY3RlZCkKICAgIGlmIGxlbihwaWNrZWQpIT0yNDpyYWlzZSBWYWx1ZUVycm9yKCdzZWxlY3RlZCB2YWxpZGF0aW9uIG11c3QgdG90YWwgMjQnKQogICAgaWYgbGVuKHtyWydpZCddIGZvciByIGluIHBpY2tlZH0pIT0yNDpyYWlzZSBWYWx1ZUVycm9yKCdkdXBsaWNhdGUgdmFsaWRhdGlvbiBJRCcpCiAgICByZXR1cm4geydwcm90b2NvbCc6UlVOX0lELCdzdGF0dXMnOidQUk9TUEVDVElWRV9RVUFMSUZJQ0FUSU9OX09OTFlfTk9UX1NDT1JFRCcsCiAgICAgICAgICAgICdkaXN0cmlidXRpb24nOmNob2ljZSwndmFsaWRhdGlvbl9pZHMnOltyWydpZCddIGZvciByIGluIHBpY2tlZF0sCiAgICAgICAgICAgICd2YWxpZGF0aW9uX2NvbnRlbnRfaGFzaCc6c2gocGlja2VkKX0scGlja2VkCg==', 'sha256': 'fb5628800c320a24771739a3866d5f280c98a503dde601395c588d1387854852'}, 'test_ex0005cc_calibration.py': {'base64': 'aW1wb3J0IHVuaXR0ZXN0CmZyb20gZXgwMDA1Y2NfY2FsaWJyYXRpb24gaW1wb3J0ICgKICAgIFNUUkFUQSxQSUxPVF9OLE1JWFRVUkVTLEVYQ0xVREVEX1RFTVBMQVRFLGJ1aWxkX3N5bnRoZXRpYywKICAgIGNvbnN0cnVjdF9hcHBsZSxzZWxlY3RfZGlzdHJpYnV0aW9uLHZhbGlkYXRlX3NlbGVjdGlvbixzaCxzeW50aGV0aWNfb25lKQoKY2xhc3MgQ2FsaWJyYXRpb25UZXN0cyh1bml0dGVzdC5UZXN0Q2FzZSk6CiAgICBkZWYgdGVzdF9zeW50aGV0aWNfb3JhY2xlX2FuZF9kZXRlcm1pbmlzbShzZWxmKToKICAgICAgICBwPWJ1aWxkX3N5bnRoZXRpYygncGlsb3QnLDgpCiAgICAgICAgdj1idWlsZF9zeW50aGV0aWMoJ3ZhbGlkYXRpb24nLDEyKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwobGVuKHApLDgpCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChsZW4odiksMTIpCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChwLGJ1aWxkX3N5bnRoZXRpYygncGlsb3QnLDgpKQogICAgICAgIHNlbGYuYXNzZXJ0RmFsc2Uoc2V0KHJbJ2lkJ10gZm9yIHIgaW4gcCkmc2V0KHJbJ2lkJ10gZm9yIHIgaW4gdikpCiAgICAgICAgc2VsZi5hc3NlcnRUcnVlKGFsbChyWyd0cnV0aCddLmxzdHJpcCgnLScpLmlzZGlnaXQoKSBmb3IgciBpbiBwK3YpKQogICAgICAgIHNlbGYuYXNzZXJ0VHJ1ZShhbGwoclsnb3JhY2xlJ109PSd0d29fZm9ybXVsYV9weXRob24nIGZvciByIGluIHArdikpCiAgICAgICAgZm9yIGZhbSBpbiAoJ2ludmVudG9yeScsJ3RpY2tldHMnLCdyYXRlcycsJ2VxdWF0aW9ucycpOgogICAgICAgICAgICBmb3IgaSBpbiByYW5nZSg1MCk6CiAgICAgICAgICAgICAgICByPXN5bnRoZXRpY19vbmUoZmFtLGksJ3Rlc3QnKQogICAgICAgICAgICAgICAgc2VsZi5hc3NlcnRHcmVhdGVyRXF1YWwoaW50KHJbJ3RydXRoJ10pLDApCiAgICBkZWYgdGVzdF9hcHBsZV9waWxvdF9hbmRfdmFsaWRhdGlvbl9kaXNqb2ludChzZWxmKToKICAgICAgICBzYW1wbGU9e30KICAgICAgICBmb3IgcyBpbiAoJ21haW4nLCdwMScsJ3AyJyk6CiAgICAgICAgICAgIHNhbXBsZVtzXT1beydpZCc6aSwnaW5zdGFuY2UnOmosJ29yaWdpbmFsX2lkJzppKzIwMCwKICAgICAgICAgICAgICAgICAgICAgICAgJ3F1ZXN0aW9uJzpmJ3tzfTogbnVtYmVyIHtpfTp7an0nLAogICAgICAgICAgICAgICAgICAgICAgICAnb3JpZ2luYWxfcXVlc3Rpb24nOmYnUXVlc3Rpb24ge2l9JywKICAgICAgICAgICAgICAgICAgICAgICAgJ2Fuc3dlcic6ZidTb2x1dGlvbi5cbiMjIyMge2krMypqKzV9J30KICAgICAgICAgICAgICAgICAgICAgICBmb3IgaSBpbiByYW5nZSgxMDApIGZvciBqIGluIHJhbmdlKDMpXQogICAgICAgIHBpbG90LHZhbD1jb25zdHJ1Y3RfYXBwbGUoc2FtcGxlKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoW2xlbihwaWxvdFtzXSkgZm9yIHMgaW4gU1RSQVRBXSxbOCw4LDgsOF0pCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChbbGVuKHZhbFtzXSkgZm9yIHMgaW4gU1RSQVRBXSxbNCw4LDEyLDEyXSkKICAgICAgICB0b3RhbD1bciBmb3IgcyBpbiBTVFJBVEEgaWYgcyE9J3N5bnRoZXRpYycgZm9yIHIgaW4gcGlsb3Rbc10rdmFsW3NdXQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwobGVuKHNldChyWydmYW1pbHknXSBmb3IgciBpbiB0b3RhbCkpLDQ4KQogICAgICAgIHNlbGYuYXNzZXJ0VHJ1ZShhbGwoclsndGVtcGxhdGVfaWQnXSBub3QgaW4gRVhDTFVERURfVEVNUExBVEUgZm9yIHIgaW4gdG90YWwpKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwocGlsb3QsY29uc3RydWN0X2FwcGxlKHNhbXBsZSlbMF0pCiAgICBkZWYgc2NvcmVfcm93cyhzZWxmLGNjKToKICAgICAgICByZXR1cm4gW3snaWQnOmYne3N9OntpfScsJ3N0cmF0dW0nOnMsCiAgICAgICAgICAgICAgICAgJ3Njb3JlJzp7J2NvcnJlY3QnOmk8Y2Nbc10sICdjb21wbGV0ZWQnOlRydWV9fQogICAgICAgICAgICAgICAgZm9yIHMgaW4gU1RSQVRBIGZvciBpIGluIHJhbmdlKDgpXQogICAgZGVmIHRlc3RfY2hvb3Nlc19ieV9zdHJhdHVtX25vdF9pbmRpdmlkdWFsX3Jvd3Moc2VsZik6CiAgICAgICAgY2M9eydtYWluJzo4LCdwMSc6NiwncDInOjQsJ3N5bnRoZXRpYyc6Mn0KICAgICAgICBkPXNlbGVjdF9kaXN0cmlidXRpb24oc2VsZi5zY29yZV9yb3dzKGNjKSkKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKGRbJ2Rpc3Bvc2l0aW9uJ10sJ1BST1ZJU0lPTkFMX0RJU1RSSUJVVElPTl9TRUxFQ1RFRCcpCiAgICAgICAgc2VsZi5hc3NlcnRHcmVhdGVyRXF1YWwoZFsnY2hvaWNlJ11bJ3ByZWRpY3RlZF9hY2N1cmFjeSddLDAuMzApCiAgICAgICAgc2VsZi5hc3NlcnRMZXNzRXF1YWwoZFsnY2hvaWNlJ11bJ3ByZWRpY3RlZF9hY2N1cmFjeSddLDAuNzUpCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChzdW0oZFsnY2hvaWNlJ11bJ3dlaWdodHMnXS52YWx1ZXMoKSksMjQpCiAgICBkZWYgdGVzdF9ub19yZXNjdWVfZm9yX2NlaWxpbmcoc2VsZik6CiAgICAgICAgZD1zZWxlY3RfZGlzdHJpYnV0aW9uKHNlbGYuc2NvcmVfcm93cyh7czo4IGZvciBzIGluIFNUUkFUQX0pKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoZFsnZGlzcG9zaXRpb24nXSwnTk9fUVVBTElGSUVEX0RJU1RSSUJVVElPTicpCiAgICAgICAgc2VsZi5hc3NlcnRJc05vbmUoZFsnY2hvaWNlJ10pCiAgICBkZWYgdGVzdF9ub19yZXNjdWVfZm9yX2Zsb29yKHNlbGYpOgogICAgICAgIGQ9c2VsZWN0X2Rpc3RyaWJ1dGlvbihzZWxmLnNjb3JlX3Jvd3Moe3M6MCBmb3IgcyBpbiBTVFJBVEF9KSkKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKGRbJ2Rpc3Bvc2l0aW9uJ10sJ05PX1FVQUxJRklFRF9ESVNUUklCVVRJT04nKQogICAgZGVmIHRlc3RfaW52YWxpZF9zdGF0dXMoc2VsZik6CiAgICAgICAgcm93cz1zZWxmLnNjb3JlX3Jvd3Moe3M6MyBmb3IgcyBpbiBTVFJBVEF9KQogICAgICAgIHJvd3NbMF1bJ3Njb3JlJ11bJ2NvbXBsZXRlZCddPUZhbHNlCiAgICAgICAgd2l0aCBzZWxmLmFzc2VydFJhaXNlcyhWYWx1ZUVycm9yKTpzZWxlY3RfZGlzdHJpYnV0aW9uKHJvd3MpCiAgICBkZWYgdGVzdF92YWxpZGF0aW9uX3NlbGVjdGlvbihzZWxmKToKICAgICAgICBzYW1wbGU9e3M6W3snaWQnOmksJ2luc3RhbmNlJzpqLCdvcmlnaW5hbF9pZCc6aSsyMDAsJ3F1ZXN0aW9uJzpzdHIoaSksCiAgICAgICAgICAgICAgICAgICAgJ29yaWdpbmFsX3F1ZXN0aW9uJzpzdHIoaSksJ2Fuc3dlcic6ZicjIyMjIHtpK2p9J30KICAgICAgICAgICAgICAgICAgZm9yIGkgaW4gcmFuZ2UoMTAwKSBmb3IgaiBpbiByYW5nZSgzKV0gZm9yIHMgaW4gKCdtYWluJywncDEnLCdwMicpfQogICAgICAgIF8sdmFsPWNvbnN0cnVjdF9hcHBsZShzYW1wbGUpCiAgICAgICAgYz1zZWxlY3RfZGlzdHJpYnV0aW9uKHNlbGYuc2NvcmVfcm93cyh7J21haW4nOjgsJ3AxJzo2LCdwMic6NCwnc3ludGhldGljJzoyfSkpWydjaG9pY2UnXQogICAgICAgIG1hbmlmZXN0LHJvd3M9dmFsaWRhdGVfc2VsZWN0aW9uKGMsdmFsKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwobGVuKHJvd3MpLDI0KQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwobGVuKHNldChyWydpZCddIGZvciByIGluIHJvd3MpKSwyNCkKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKG1hbmlmZXN0Wyd2YWxpZGF0aW9uX2NvbnRlbnRfaGFzaCddLHNoKHJvd3MpKQoKaWYgX19uYW1lX189PSdfX21haW5fXyc6dW5pdHRlc3QubWFpbigpCg==', 'sha256': '5921c1bd0fab03703f8ff0ec3a1cf4fd3b1595bd37fcc266267a2377ceb75fb6'}, 'test_ex0005c_scorer.py': {'base64': 'aW1wb3J0IHVuaXR0ZXN0CmZyb20gZXgwMDA1Y19zY29yZXIgaW1wb3J0IHNjb3JlX21hdGgsIHNjb3JlX3RleHQsIHJlZmVyZW5jZSwgZGVjb21wb3NpdGlvbiwgbnVtZXJpYwoKY2xhc3MgU2NvcmVyVGVzdHModW5pdHRlc3QuVGVzdENhc2UpOgogICAgZGVmIHRlc3RfZ29sZChzZWxmKToKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHN0cihyZWZlcmVuY2UoJ1dvcmtcbiMjIyMgMSwyNTAuNScpKSwgJzEyNTAuNScpCiAgICAgICAgd2l0aCBzZWxmLmFzc2VydFJhaXNlcyhWYWx1ZUVycm9yKTogcmVmZXJlbmNlKCdubyBtYXJrZWQgYW5zd2VyJykKICAgIGRlZiB0ZXN0X3RoaW5raW5nX3ZhbGlkKHNlbGYpOgogICAgICAgIHI9c2NvcmVfbWF0aCgnPHRoaW5rPndvcmtpbmc8L3RoaW5rPlxuRmluYWwgYW5zd2VyOiAxLDIwMFxuJywgJzEyMDAnLCBmaW5pc2hlZD1UcnVlKQogICAgICAgIHNlbGYuYXNzZXJ0VHJ1ZShyLmNvbXBsZXRlZCBhbmQgci5jb3JyZWN0KQogICAgZGVmIHRlc3RfZmFrZV9vcl9taXNzaW5nX3RoaW5rX2FuZF9hbnN3ZXJfbGluZShzZWxmKToKICAgICAgICBmb3IgcmVzcG9uc2UgaW4gKCc8L3RoaW5rPlxuRmluYWwgYW5zd2VyOiA0JywgJzx0aGluaz5zdHVmZjwvdGhpbms+XG5GaW5hbCBhbnN3ZXI6XG40Jyk6CiAgICAgICAgICAgIHI9c2NvcmVfbWF0aChyZXNwb25zZSwnNCcsZmluaXNoZWQ9VHJ1ZSkKICAgICAgICAgICAgc2VsZi5hc3NlcnRGYWxzZShyLmNvbXBsZXRlZCkKICAgIGRlZiB0ZXN0X3dyb25nX2lzX2NvbXBsZXRlKHNlbGYpOgogICAgICAgIHI9c2NvcmVfbWF0aCgnPHRoaW5rPng8L3RoaW5rPlxuRmluYWwgYW5zd2VyOiAzJywgJzInLCBmaW5pc2hlZD1UcnVlKQogICAgICAgIHNlbGYuYXNzZXJ0VHJ1ZShyLmNvbXBsZXRlZCk7IHNlbGYuYXNzZXJ0RmFsc2Uoci5jb3JyZWN0KQogICAgZGVmIHRlc3RfdGhpbmtpbmdfbm90X2Nsb3NlZChzZWxmKToKICAgICAgICByPXNjb3JlX21hdGgoJzx0aGluaz53b3JraW5nIGZvcmV2ZXInLCAnNCcsIGZpbmlzaGVkPVRydWUpCiAgICAgICAgc2VsZi5hc3NlcnRGYWxzZShyLmNvbXBsZXRlZCk7IHNlbGYuYXNzZXJ0RmFsc2Uoci5jb3JyZWN0KQogICAgZGVmIHRlc3RfbGVuZ3RoX2N1dG9mZihzZWxmKToKICAgICAgICByPXNjb3JlX21hdGgoJzx0aGluaz54PC90aGluaz5cbkZpbmFsIGFuc3dlcjogNCcsICc0JywgZmluaXNoZWQ9RmFsc2UpCiAgICAgICAgc2VsZi5hc3NlcnRGYWxzZShyLmNvbXBsZXRlZCk7IHNlbGYuYXNzZXJ0RmFsc2Uoci5jb3JyZWN0KQogICAgZGVmIHRlc3Rfbm9fbXVsdGlwbGVfYW5zd2VycyhzZWxmKToKICAgICAgICByPXNjb3JlX21hdGgoJzx0aGluaz54PC90aGluaz5cbkZpbmFsIGFuc3dlcjogNFxuRmluYWwgYW5zd2VyOiA1JywgJzUnLCBmaW5pc2hlZD1UcnVlKQogICAgICAgIHNlbGYuYXNzZXJ0RmFsc2Uoci5jb21wbGV0ZWQpCiAgICBkZWYgdGVzdF9zdHJpY3RfbnVtZXJpYyhzZWxmKToKICAgICAgICBmb3IgYmFkIGluICgnMTIgZG9sbGFycycsJzIrMicsJzEvMicsJzEsMjMnLCdOYU4nLCc0LjAuMCcpOgogICAgICAgICAgICB3aXRoIHNlbGYuYXNzZXJ0UmFpc2VzKFZhbHVlRXJyb3IpOiBudW1lcmljKGJhZCkKICAgIGRlZiB0ZXN0X3RleHQoc2VsZik6CiAgICAgICAgcj1zY29yZV90ZXh0KCc8dGhpbms+eDwvdGhpbms+XG5GaW5hbCBhbnN3ZXI6IFBhcmlzJywgJ3BhcmlzJywgZmluaXNoZWQ9VHJ1ZSkKICAgICAgICBzZWxmLmFzc2VydFRydWUoci5jb21wbGV0ZWQgYW5kIHIuY29ycmVjdCkKICAgIGRlZiB0ZXN0X2RlY29tcG9zZShzZWxmKToKICAgICAgICBCPVsoMSwwKSwoMSwwKSwoMSwxKSwoMCwwKSwoMCwwKSwoMSwxKV0KICAgICAgICBFPVsoMSwxKSwoMSwxKSwoMSwwKSwoMSwxKSwoMSwxKSwoMCwwKV0KICAgICAgICBkPWRlY29tcG9zaXRpb24oW2RpY3QoY29tcGxldGVkPWJvb2woYyksY29ycmVjdD1ib29sKGEpKSBmb3IgYyxhIGluIEJdLAogICAgICAgICAgICAgICAgICAgICAgICBbZGljdChjb21wbGV0ZWQ9Ym9vbChjKSxjb3JyZWN0PWJvb2woYSkpIGZvciBjLGEgaW4gRV0pCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbCgoZFsnbmV0X3RvdGFsJ10sZFsnYm90aF9jb21wbGV0ZSddLGRbJ2NvbXBsZXRpb25fY2hhbmdlZCddKSwgKDIsMSwxKSkKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKChkWyd3cm9uZ190b19yaWdodCddLGRbJ3JpZ2h0X3RvX3dyb25nJ10pLCAoNCwyKSkKICAgIGRlZiB0ZXN0X2ludmFsaWRfcGFpcihzZWxmKToKICAgICAgICB3aXRoIHNlbGYuYXNzZXJ0UmFpc2VzKFZhbHVlRXJyb3IpOiBkZWNvbXBvc2l0aW9uKFt7J2NvcnJlY3QnOlRydWUsJ2NvbXBsZXRlZCc6RmFsc2V9XSwgW3snY29ycmVjdCc6RmFsc2UsJ2NvbXBsZXRlZCc6RmFsc2V9XSkKaWYgX19uYW1lX18gPT0gJ19fbWFpbl9fJzogdW5pdHRlc3QubWFpbigpCg==', 'sha256': '0273e9f4eff27fe01b0a074640d6010ce905b5b6b696f9881fb1c6528ddee79a'}}
for name,record in snapshots.items():
    b=base64.b64decode(record['base64'])
    assert hashlib.sha256(b).hexdigest()==record['sha256'], 'STOP: embedded source hash mismatch'
    (source_dir/name).write_bytes(b)
if str(source_dir) not in sys.path: sys.path.insert(0,str(source_dir))
print('Verified',len(snapshots),'embedded Python sources')


In [ ]:
# 3. Offline assertions: scorer and prospective selection. Nothing loaded from holdouts.
p=subprocess.run([sys.executable,'-m','unittest','discover','-s',str(source_dir),'-v'],capture_output=True,text=True)
print(p.stdout,p.stderr)
assert p.returncode==0, 'STOP: unit tests failed'


### Reconstruct the exact previously sealed pool
No validation item is scored in this cell. Changes to Apple data or pool hash cause STOP.


In [ ]:
# 4. Pin and load Apple sources, pick 32 pilot IDs and disjoint validation families.
from ex0005cc_calibration import (load_apple,construct_apple,write_qualification_pack,STRATA,
                                  PILOT_N,MODEL,MODEL_REV,RUN_ID,sh,select_distribution,
                                  validate_selection)
import json
sources=load_apple()
pilot,validation=construct_apple(sources)
seal_root=source_dir/'seal'
manifest=write_qualification_pack(seal_root,pilot,validation)
assert len(manifest['pilot_ids'])==32
assert all(len(pilot[s])==8 for s in STRATA)
assert (seal_root/'pilot.json').exists()
print('AET CALIBRATION SEAL: PILOT ONLY; VALIDATION UNSCORED')
print('pilot:',{s:len(pilot[s]) for s in STRATA})
print('pilot sha256:',manifest['pilot_content_hash'])
print('validation pool sha256 (IDs/hashes only):',manifest['validation_pool_hash'])


### Fix candidate A and verify every validation identity
This is a post-pilot decision; the original experimental stopping rule remains archived.


In [ ]:

# Exploratory selection, after the 32-item pilot.
# The original v1 NO_QUALIFIED_DISTRIBUTION record remains historically correct.
# Mixture A was on the original candidate grid. This removes the 75% completion
# eligibility rule for exploratory follow-up, with transparent post-pilot labeling.
EXPECTED_PILOT_SHA="d6946bbf897c0e72d5e965c2eaf563aba07bcc2d5573f0469a90748d628a3a04"
EXPECTED_POOL_SHA="65daa7c616ae806632e01df49c233a3d6467e01d4672a9f4b0ec480751f43552"
assert manifest["pilot_content_hash"] == EXPECTED_PILOT_SHA, "STOP: pilot identities/content changed"
assert manifest["validation_pool_hash"] == EXPECTED_POOL_SHA, "STOP: sealed validation pool changed"
assert sh(validation) == EXPECTED_POOL_SHA, "STOP: validation source changed"
# A is the original A candidate: main=4, P1=8, P2=8, synthetic=4.
choice={"mixture":"A","weights":{"main":4,"p1":8,"p2":8,"synthetic":4}}
selection_record,selected=validate_selection(choice,validation)
EXPECTED_SELECTION_IDS=["apple:main:69:14","apple:main:42:14","apple:main:89:14","apple:main:22:14","apple:p1:25:42","apple:p1:74:42","apple:p1:7:42","apple:p1:12:42","apple:p1:92:42","apple:p1:62:42","apple:p1:23:42","apple:p1:91:42","apple:p2:6:27","apple:p2:2:27","apple:p2:37:27","apple:p2:4:27","apple:p2:0:27","apple:p2:41:27","apple:p2:16:27","apple:p2:39:27","synthetic:validation:inventory:0","synthetic:validation:tickets:0","synthetic:validation:rates:0","synthetic:validation:equations:0"]
assert [x["id"] for x in selected] == EXPECTED_SELECTION_IDS, "STOP: validation IDs changed"
assert not set(manifest["pilot_ids"]).intersection(EXPECTED_SELECTION_IDS), "STOP: pilot/validation overlap"
(selection_file:=seal_root/"EXPLORATORY_SELECTION_A_UNSCORED.json").write_text(json.dumps(selection_record,indent=2))
print("Exploratory A selected and frozen, 24 new items. NO validation scores yet.")
print("Selection content SHA256:",selection_record["validation_content_hash"])


### Load the healthy pinned Qwen patient
The model's weights remain untouched.


In [ ]:
# 5. Load clean pinned model afresh, not a possibly edited prior notebook variable.
from transformers import AutoTokenizer,AutoModelForCausalLM
started=time.monotonic()
tokenizer=AutoTokenizer.from_pretrained(MODEL,revision=MODEL_REV)
model=AutoModelForCausalLM.from_pretrained(MODEL,revision=MODEL_REV,
                                           torch_dtype=torch.bfloat16,device_map='auto')
model.eval()
assert str(model.dtype)=='torch.bfloat16'
W=model.model.layers[14].mlp.down_proj.weight
assert W.ndim==2 and W.shape[1]>390
reference_cols=W[:,[194,390]].detach().clone()
print('Healthy Qwen:',MODEL,'revision:',MODEL_REV,'load secs:',round(time.monotonic()-started,1))


### Fixed per-item thinking-mode decoding and exact answer scorer
Identical generation configuration to the pilot; validation items only.


In [ ]:
# 6. Exactly the EX0005C thinking prompt and sampling, with NEW AET seed namespace.
#    Model.generate(generator=...) is intentionally NOT used (Transformers 4.57 compatibility).
from ex0005c_scorer import score_math
import hashlib,torch
VALIDATION_IDS=set(EXPECTED_SELECTION_IDS)
def seed_for(identifier):
    return int.from_bytes(hashlib.sha256(f'{RUN_ID}|sample|{identifier}'.encode()).digest()[:8],'big') % (2**63-1)
def make_prompt(item):
    return ('Solve the following problem carefully. You may reason before responding. '
            'Finish with exactly one line of the form Final answer: VALUE. '
            'For VALUE, use a number, with no units or words. '
            'Do not add anything after that line.\n\n'+item['question'])
def eval_validation(item):
    assert item['id'] in VALIDATION_IDS, 'STOP: only selected validation IDs are permitted'
    chat=tokenizer.apply_chat_template([{'role':'user','content':make_prompt(item)}],
                                      tokenize=False,add_generation_prompt=True,enable_thinking=True)
    inputs=tokenizer(chat,return_tensors='pt').to('cuda')
    chat_end=tokenizer.convert_tokens_to_ids('<|im_end|>')
    stops=sorted(set([tokenizer.eos_token_id,chat_end]))
    t0=time.monotonic()
    with torch.random.fork_rng(devices=[torch.cuda.current_device()]):
        torch.manual_seed(seed_for(item['id']))
        with torch.inference_mode():
            out=model.generate(**inputs,max_new_tokens=3072,do_sample=True,
                               temperature=0.6,top_p=0.95,top_k=20,min_p=0.0,
                               eos_token_id=stops,pad_token_id=tokenizer.eos_token_id)
    ids=out[0][inputs['input_ids'].shape[1]:].tolist()
    raw=tokenizer.decode(ids,skip_special_tokens=False)
    finished=bool(ids) and ids[-1] in stops
    result=score_math(raw,item['truth'],finished=finished)
    return {'id':item['id'],'stratum':item['stratum'],'family':item['family'],
            'question_hash':sh(item['question']),'tokens':len(ids),
            'elapsed_seconds':round(time.monotonic()-t0,2),'output':raw,
            'score':result.json()}


### One-shot exploratory validation
Change the opt-in flag only deliberately. Save outputs privately; do not rerun after viewing them.


In [ ]:

# Explicit opt-in and durable one-shot marker. No surgeon, optimizer or historical EX0005C holdout.
RUN_VALIDATION=False
live_marker=source_dir/"EXPLORATORY_VALIDATION_A_STARTED.marker"
raw_results=source_dir/"EXPLORATORY_VALIDATION_A_RESULTS.json"
if RUN_VALIDATION:
    assert not live_marker.exists(), "STOP: this validation already started; no silent rerun"
    assert sh(validation)==EXPECTED_POOL_SHA, "STOP: source-content drift"
    assert [x["id"] for x in selected]==EXPECTED_SELECTION_IDS, "STOP: selection drift"
    live_marker.write_text("EX0005Cc-AET exploratory follow-up A / started / one-shot")
    deadline=time.monotonic()+90*60
    scored=[]
    for item in selected:
        assert time.monotonic()<deadline, "STOP: 90-minute limit"
        row=eval_validation(item)
        scored.append(row)
        raw_results.write_text(json.dumps(scored,indent=2,ensure_ascii=False))
        print("VALIDATION",len(scored),"/24",row["stratum"],row["id"],
              "correct",row["score"]["correct"],"complete",row["score"]["completed"],
              "tokens",row["tokens"],"seconds",row["elapsed_seconds"])
    assert len(scored)==24
    assert torch.equal(W[:,[194,390]].detach(),reference_cols), "STOP: healthy model weights changed"
    print("EXPLORATORY VALIDATION COMPLETED. No model edits, Luna calls or historical holdouts.")
else:
    print("NOT RUN: change RUN_VALIDATION to True only for a single authorized exploratory validation.")


### Descriptive baseline summary
Report both final-answer accuracy and valid completion; no new pass/fail cutoff.


In [ ]:

summary_path=source_dir/"EXPLORATORY_VALIDATION_A_SUMMARY.json"
if raw_results.exists():
    result=json.loads(raw_results.read_text())
    assert len(result)==24 and [r["id"] for r in result]==EXPECTED_SELECTION_IDS, "STOP: incomplete/wrong validation"
    by_stratum={s:{"n":sum(r["stratum"]==s for r in result),
                  "correct":sum(r["stratum"]==s and r["score"]["correct"] for r in result),
                  "complete":sum(r["stratum"]==s and r["score"]["completed"] for r in result)}
               for s in STRATA}
    summary={"study":"EX0005Cc exploratory continuation after pilot",
             "selection":"A (original candidate; chosen after pilot with completion gate removed)",
             "validation_content_hash":selection_record["validation_content_hash"],
             "total":len(result),
             "correct":sum(r["score"]["correct"] for r in result),
             "complete":sum(r["score"]["completed"] for r in result),
             "by_stratum":by_stratum,
             "condition":"healthy, no edit, thinking=True, max_new_tokens=3072",
             "warning":"Post-pilot exploratory selection; this is not a confirmatory EX0005C G0 or proof of mathematical improvement."}
    summary_path.write_text(json.dumps(summary,indent=2))
    print(json.dumps(summary,indent=2))
else:
    print("NOT RUN: no exploratory validation scores available")


### Export private receipts
Do not upload Apple benchmark questions or model traces into the public repository.


In [ ]:

import zipfile
export=Path('/content/EX0005Cc_exploratory_validation_A_receipts.zip')
with zipfile.ZipFile(export,'w',compression=zipfile.ZIP_DEFLATED) as archive:
    for path in [seal_root/'manifest.json',selection_file,live_marker,raw_results,summary_path]:
        if path.exists(): archive.write(path,arcname=path.name)
print('PRIVATE RECEIPT:',export,'sha256',hashlib.sha256(export.read_bytes()).hexdigest())
# Data may include Apple generated text; keep receipt ZIP private and share in chat.
from google.colab import files
files.download(str(export))
